# logit-thermo A2-10：凝固点降下の分析（CPU）

- **必要な secret：なし**。**ランタイム：CPU（標準）**。見込みは 10〜20 分。**3 本のデータ集めが全部終わってから**回してください。
- `A2_10/` の 3 モデル × 5 本の腕を読み、T_m（凍る速さと溶ける速さ）と T_0（ℓ の交点）を出して判定し、`A2_10/a2_10_summary.json` に書きます。**結果は表示しません。**
- 計算のコード：A2-2・A2-4・A2-7b の分析、結晶の始まりの探索、A2-8 のコードは Drive の前のノートブックから取り出してハッシュで確かめ、`scripts/a2_10_analyze.py`（`eb42efb874c6b32d…`）は埋め込み（データを開く前に固定）。
- **通知**：🚀・🎉・⚠️。名札は「logit-thermo A2-10 凝固点降下 分析（T_m と T_0 の判定、CPU）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo A2-10 凝固点降下 分析（T_m と T_0 の判定、CPU）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - A2-10 analyze', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：分析のコード
import ast, glob, hashlib, json
def take_src(want):
    """want：{変数の名前: sha256}。Drive の logit-thermo のノートブックから、埋め込んだコードを取り出してハッシュで確かめる。"""
    got = {}
    for p in sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/*.ipynb', recursive=True)):
        try:
            cells = json.load(open(p))['cells']
        except Exception:
            continue
        for c in cells:
            s = c['source'] if isinstance(c['source'], str) else ''.join(c['source'])
            if c.get('cell_type') != 'code' or 'SRC' not in s:
                continue
            try:
                body = ast.parse(s).body
            except SyntaxError:
                continue
            for n in body:
                if isinstance(n, ast.Assign) and isinstance(n.targets[0], ast.Name) and n.targets[0].id in want and n.targets[0].id not in got:
                    v = ast.literal_eval(n.value)
                    if hashlib.sha256(v.encode()).hexdigest() == want[n.targets[0].id]:
                        got[n.targets[0].id] = v
        if len(got) == len(want):
            break
    miss = [k for k in want if k not in got]
    assert not miss, ('Drive のノートブックからコードを取り出せませんでした（前のノートブックを動かしていませんか）：', miss)
    return got
_s = take_src({'SRCA22': 'fe1f96db1f7e84031e85a70bcc8dcbb95c9fc06b3367657ff4074a9f9d1343f7', 'SRCA24': 'e70bad0db9b7780c2059aae52ad06debe87757bc582791deefb58c5f435f84a6', 'SRC7BA': 'd4da9ac740e2b6451845da8496ff90e451d5ba11bbcef28e1206da2cc61d1135', 'SRCON': '0b39f997b72f33d3903df735b9e6525b9786cc714e4457e4442a2d6db1c803e9', 'SRC8': '06a93c4eec9f3e770ca9e2f3f791a94b37153c908becd0f83d825acfbd11bacd'})
SRCA22, SRCA24, SRC7BA, SRCON, SRC8 = _s['SRCA22'], _s['SRCA24'], _s['SRC7BA'], _s['SRCON'], _s['SRC8']
SRC10A = '"""A2-10（ゲート docs/A2_10_GATE.md）の分析。CPU。データを開く前に固定する。\nscripts/a2_2_analyze.py（if False: 版。crossing_mono・per_doc_feedback・feedback_stats）、scripts/a2_4_analyze.py（stats_fixed）、\nscripts/a2_7b_analyze.py（slope0・log_q）、scripts/a2_explore_onset.py（copy_marks）、scripts/a2_8_vh.py（phases・doc_counts・fit_vh）を\n実行したのと同じ名前空間で実行する。fit_vh が読む温度の並び A8_TU は、ここで A2-10 の 14 の温度に置きかえる。\n"""\nimport math\nimport numpy as np\n\nC10A_TEMPS = [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]\nC10A_T0_TEMPS = [0.6, 0.7, 0.8, 0.85, 0.9, 0.95, 1.0, 1.05]      # T_0 はこの 8 つだけで（A2-7c と同じ格子。ε が低い温度でふくらまないように）\nC10A_MODELS = [\'pythia-1b\', \'olmo2-1b\', \'qwen2.5-1.5b\']\nC10A_ARMS = [\'none\', \'none_b\', \'pl075\', \'pl150\', \'pl300\']\nC10A_PL = [\'pl075\', \'pl150\', \'pl300\']\nC10A_DOSE = {\'none\': 0.0, \'none_b\': 0.0, \'pl075\': 0.075, \'pl150\': 0.15, \'pl300\': 0.30}\nC10A_BOOT, C10A_SEED = 1000, 20261117\nC10A_R = 1.5\nC10A_Q1_BINS = [(0.05, \'ほとんど下がらない（log で 0.05 未満、5% 未満）\'), (0.15, \'少し下がる（log で 0.05〜0.15、5〜14%）\'),\n                (0.30, \'かなり下がる（log で 0.15〜0.30、14〜26%）\'), (float(\'inf\'), \'大きく下がる（log で 0.30 以上、26% 以上）\')]\nA8_TU = C10A_TEMPS\n\n\ndef counts_arm10(toks):\n    """toks：(14, S, nD, L) → (14, nD, 10)（U・HT は 0。T_m だけ出す）。"""\n    nT, S, nD, L = toks.shape\n    out = np.zeros((nT, nD, 10))\n    z = np.zeros((S, L))\n    for k in range(nT):\n        for j in range(nD):\n            C = np.stack([copy_marks(toks[k, s, j]) for s in range(S)])\n            out[k, j] += doc_counts(C, z, z)\n    return out\n\n\ndef load10(d, m):\n    Zt = np.load(f\'{d}/{m}@tf.npz\')\n    assert [float(t) for t in Zt[\'temps\']] == C10A_TEMPS\n    i0 = [C10A_TEMPS.index(t) for t in C10A_T0_TEMPS]\n    tf = Zt[\'tf\'][:, i0]\n    fds, cnt, mass, top1 = {}, {}, {}, {}\n    for a in C10A_ARMS:\n        Z = np.load(f\'{d}/{m}@{a}.npz\')\n        assert [float(t) for t in Z[\'temps\']] == C10A_TEMPS\n        fds[a] = per_doc_feedback(tf, Z[\'g_ell\'][i0])\n        cnt[a] = counts_arm10(Z[\'toks\'])\n        bm = Z[\'g_ban_mass\'].astype(np.float64)\n        mass[a] = bm.mean(axis=(1, 2, 3)).tolist()                 # 温度ごとの、取り上げた確率の量\n        top1[a] = Z[\'g_banned_top1\'].astype(np.float64).mean(axis=(1, 2, 3)).tolist()\n    return fds, cnt, mass, top1\n\n\ndef fit_once10(fds, cnt, eps, w, use):\n    """w：記事の重み（選び直し）。use：{腕: 使う温度の番号}（点のときに、その腕の凍る回・溶ける回がそれぞれ 10 回以上の温度）。\n    → {腕: (y_m, y_0)}、全部の腕で T_m と T_0 が出たか。"""\n    idx = np.repeat(np.arange(len(w)), w.astype(int))\n    T0n = crossing_mono(np.median(fds[\'none\'][idx], 0), C10A_T0_TEMPS, eps)\n    fm = {a: fit_vh(np.tensordot(cnt[a], w, axes=([1], [0])), temps_use=use[a]) for a in C10A_ARMS}\n    Tmn = fm[\'none\'][\'Tm\'] if fm[\'none\'][\'ok\'] else float(\'nan\')\n    out, ok_m, ok_0 = {}, True, True\n    for a in C10A_ARMS:\n        if a == \'none\':\n            continue\n        tm = fm[a][\'Tm\'] if fm[a][\'ok\'] else float(\'nan\')\n        t0 = crossing_mono(np.median(fds[a][idx], 0), C10A_T0_TEMPS, eps)\n        ym = math.log(tm / Tmn) if np.isfinite(tm) and np.isfinite(Tmn) else float(\'nan\')\n        y0 = math.log(t0 / T0n) if np.isfinite(t0) and np.isfinite(T0n) else float(\'nan\')\n        ok_m &= np.isfinite(ym); ok_0 &= np.isfinite(y0)\n        out[a] = (ym, y0)\n    return out, ok_m, ok_0\n\n\ndef slopes10(out):\n    """偽薬の 3 本で、原点を通る直線の傾き K_m・K_0（y ＝ −K × 濃さ）。T_0 は出た腕だけ（2 本未満なら nan）。"""\n    xm = [C10A_DOSE[a] for a in C10A_PL if np.isfinite(out[a][0])]\n    ym = [out[a][0] for a in C10A_PL if np.isfinite(out[a][0])]\n    x0 = [C10A_DOSE[a] for a in C10A_PL if np.isfinite(out[a][1])]\n    y0 = [out[a][1] for a in C10A_PL if np.isfinite(out[a][1])]\n    Km = slope0(xm, ym) if len(xm) == 3 else float(\'nan\')\n    K0 = slope0(x0, y0) if len(x0) >= 2 else float(\'nan\')\n    return Km, K0, x0\n\n\ndef q1_word10(y, ci):\n    if not (np.isfinite(y) and ci is not None and np.isfinite(ci[0])):\n        return \'はっきりしない\'\n    if ci[1] >= 0:\n        return \'はっきりしない（0 をまたぐ）\' if ci[0] < 0 else \'上がる\'\n    for edge, w in C10A_Q1_BINS:\n        if -y < edge:\n            return w\n\n\ndef q2_word10(lr, ci, fail):\n    if fail > 0.1 or not np.isfinite(lr) or ci is None:\n        return \'はっきりしない\'\n    lo, hi = ci\n    if hi < 0 and lr <= -math.log(C10A_R) + 1e-12:\n        return \'T_m のほうが速く下がる\'\n    if lo >= -math.log(C10A_R) and hi <= math.log(C10A_R):\n        return \'同じくらい\'\n    if lo > 0 and lr >= math.log(C10A_R) - 1e-12:\n        return \'T_0 のほうが速く下がる（海水型）\'\n    return \'はっきりしない\'\n\n\ndef analyze10(d):\n    res = {\'models\': {}, \'temps\': C10A_TEMPS, \'T0_temps\': C10A_T0_TEMPS}\n    for m in C10A_MODELS:\n        fds, cnt, mass, top1 = load10(d, m)\n        eps = feedback_stats(fds[\'none\'], C10A_T0_TEMPS, np.random.default_rng(0))[\'eps\']\n        use = {a: fit_vh(cnt[a].sum(1))[\'use_idx\'] for a in C10A_ARMS}   # 濃い偽薬では T_m が下がり、使える温度がずれるので腕ごとに\n        nD = cnt[\'none\'].shape[1]\n        out, okm, ok0 = fit_once10(fds, cnt, eps, np.ones(nD), use)\n        Km, K0, x0 = slopes10(out)\n        lr = log_q(K0, Km)                                          # log(K_0 ÷ K_m)\n        row = dict(eps=eps, use_temps={a: [C10A_TEMPS[i] for i in use[a]] for a in C10A_ARMS}, K_m=Km, K_0=K0, logR=lr, arms_T0=x0,\n                   arms={a: dict(dose=C10A_DOSE[a], mass_by_T=mass[a], top1_by_T=top1[a],\n                                 Tm=(fit_vh(cnt[a].sum(1), temps_use=use[a]).get(\'Tm\')), T0=stats_fixed(fds[a], C10A_T0_TEMPS, eps)[\'T0\'],\n                                 dH_vH=fit_vh(cnt[a].sum(1), temps_use=use[a]).get(\'dH_vH\'),\n                                 y_m=(out[a][0] if a != \'none\' else 0.0), y_0=(out[a][1] if a != \'none\' else 0.0)) for a in C10A_ARMS})\n        rng = np.random.default_rng(C10A_SEED)\n        b_ym, b_lr, fail_m, fail_r = {a: [] for a in C10A_ARMS if a != \'none\'}, [], 0, 0\n        for _ in range(C10A_BOOT):\n            w = np.bincount(rng.integers(0, nD, nD), minlength=nD).astype(float)\n            o, om, o0 = fit_once10(fds, cnt, eps, w, use)\n            if not om:\n                fail_m += 1\n            for a in b_ym:\n                if np.isfinite(o[a][0]):\n                    b_ym[a].append(o[a][0])\n            km, k0, _ = slopes10(o)\n            v = log_q(k0, km)\n            if not (np.isfinite(km) and np.isfinite(k0)):\n                fail_r += 1\n            elif not np.isnan(v):\n                b_lr.append(v)\n        pc = lambda v: ([float(np.percentile(v, 2.5, method=\'lower\')), float(np.percentile(v, 97.5, method=\'higher\'))] if len(v) >= 20 else None)\n        row[\'ci_y_m\'] = {a: pc(v) for a, v in b_ym.items()}\n        row[\'ci_logR\'] = pc(b_lr)\n        row[\'boot_fail_m\'], row[\'boot_fail_R\'] = fail_m / C10A_BOOT, fail_r / C10A_BOOT\n        row[\'Q1_word\'] = q1_word10(out[\'pl300\'][0], row[\'ci_y_m\'][\'pl300\']) if row[\'boot_fail_m\'] <= 0.1 else \'はっきりしない\'\n        row[\'Q2_word\'] = q2_word10(lr, row[\'ci_logR\'], row[\'boot_fail_R\'])\n        # 記録：凝固点降下の式から逆算した ΔH（x ＝ 濃さ、x ＝ T_m のまわりで取り上げた確率の量）\n        Tmn = fit_vh(cnt[\'none\'].sum(1), temps_use=use[\'none\']).get(\'Tm\')\n        if np.isfinite(Km) and Km > 0 and Tmn and np.isfinite(Tmn):\n            row[\'dH_col_dose\'] = Tmn / Km\n            mm = [float(np.mean([mass[a][i] for i in use[a]])) for a in C10A_PL]\n            Kmass = slope0(mm, [out[a][0] for a in C10A_PL])\n            row[\'dH_col_mass\'] = Tmn / Kmass if Kmass > 0 else float(\'nan\')\n        res[\'models\'][m] = row\n    ys = [res[\'models\'][m][\'arms\'][\'pl300\'][\'y_m\'] for m in C10A_MODELS]\n    words = [res[\'models\'][m][\'Q1_word\'] for m in C10A_MODELS]\n    ok = [y for y, w in zip(ys, words) if not w.startswith(\'はっきりしない\')]\n    if len(ok) >= 2:\n        med = float(np.median(ok))\n        res[\'Q1\'] = next(w for e, w in C10A_Q1_BINS if -med < e) if med < 0 else \'上がる\'\n    else:\n        res[\'Q1\'] = \'はっきりしない\'\n    res[\'Q2\'] = sum(res[\'models\'][m][\'Q2_word\'] == \'T_m のほうが速く下がる\' for m in C10A_MODELS)\n    return res\n\n\ndef self_test_a10():\n    """作り物：T_m は濃さ 30% で −20%（K_m ≈ 0.74）、T_0 は −3%（K_0 ≈ 0.1）→ Q1「かなり」、Q2「T_m のほうが速く」。"""\n    import tempfile\n    rng = np.random.default_rng(0)\n    L, nD, S = 512, 40, 2\n    t = np.array(C10A_T0_TEMPS)\n\n    def chain(Tm):\n        toks = np.zeros((len(C10A_TEMPS), S, nD, L), np.int32)\n        for k, T in enumerate(C10A_TEMPS):\n            lnK = 5.0 / T - 5.0 / Tm\n            km = 0.01 * math.exp(-lnK / 2); kf = 0.01 * math.exp(lnK / 2)\n            for s in range(S):\n                for j in range(nD):\n                    st, seq, per = 0, [], list(rng.integers(100, 30000, 7))\n                    for tt in range(L):\n                        if tt > 0:\n                            st = (1 if rng.random() < kf else 0) if st == 0 else (0 if rng.random() < km else 1)\n                        seq.append(per[tt % 7] if st else int(rng.integers(100, 30000)))\n                    toks[k, s, j] = seq\n        return toks\n    with tempfile.TemporaryDirectory() as dd:\n        for m in C10A_MODELS:\n            np.savez(f\'{dd}/{m}@tf.npz\', tf=np.zeros((nD, len(C10A_TEMPS), 24), np.float32), temps=np.array(C10A_TEMPS))\n            for a in C10A_ARMS:\n                x = C10A_DOSE[a]\n                Tm, T0 = 0.56 * math.exp(-0.74 * x), 0.86 * math.exp(-0.1 * x)\n                F = np.where(np.array(C10A_TEMPS) < T0, (np.array(C10A_TEMPS) - T0) * 12, (np.array(C10A_TEMPS) - T0) * 5)\n                g = np.broadcast_to(F[:, None, None, None], (len(C10A_TEMPS), S, nD, 24)) + rng.normal(0, 0.15, (len(C10A_TEMPS), S, nD, 24))\n                np.savez(f\'{dd}/{m}@{a}.npz\', temps=np.array(C10A_TEMPS), toks=chain(Tm), g_ell=g.astype(np.float32),\n                         g_ban_mass=np.full((len(C10A_TEMPS), S, nD, 24), x * 0.5, np.float32), g_banned_top1=np.zeros((len(C10A_TEMPS), S, nD, 24), bool))\n            break\n        # 3 モデル分は時間がかかるので、1 モデル分を名前だけ変えて置く\n        import shutil, glob, os\n        for m in C10A_MODELS[1:]:\n            for p in glob.glob(f\'{dd}/{C10A_MODELS[0]}@*.npz\'):\n                shutil.copy(p, p.replace(C10A_MODELS[0], m))\n        r = analyze10(dd)\n    row = r[\'models\'][\'pythia-1b\']\n    assert row[\'Q1_word\'] == \'かなり下がる（log で 0.15〜0.30、14〜26%）\', (row[\'arms\'][\'pl300\'], row[\'ci_y_m\'])\n    assert row[\'Q2_word\'] == \'T_m のほうが速く下がる\', (row[\'K_m\'], row[\'K_0\'], row[\'ci_logR\'])\n    assert r[\'Q1\'] == \'かなり下がる（log で 0.15〜0.30、14〜26%）\' and r[\'Q2\'] == 3, (r[\'Q1\'], r[\'Q2\'])\n    assert q2_word10(math.log(2), [0.2, 1.0], 0) == \'T_0 のほうが速く下がる（海水型）\' and q2_word10(0.0, [-0.2, 0.3], 0) == \'同じくらい\'\n    return \'OK\'\n'
assert hashlib.sha256(SRC10A.encode()).hexdigest() == 'eb42efb874c6b32dffc4a837c35e53f4c589462e5f71ae98b9f4c3c59d010e65'
exec(SRCA22)
exec(SRCA24)
exec(SRC7BA)
exec(SRCON)
exec(SRC8)
print('自己テスト（前の部品）：', self_test_a27b_analyze(), self_test_onset(), self_test_a8())
exec(SRC10A)
print('自己テスト（A2-10）：', self_test_a10())


In [ ]:
# セル3：分析して保存（結果は表示しない）
import json
try:
    r = analyze10('/content/drive/MyDrive/logit-thermo/A2_10')
    r['a2_10_analyze_sha256'] = 'eb42efb874c6b32dffc4a837c35e53f4c589462e5f71ae98b9f4c3c59d010e65'
    json.dump(_clean(r), open('/content/drive/MyDrive/logit-thermo/A2_10' + '/a2_10_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ A2-10 の分析で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：判定を A2_10/a2_10_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
